# 19 — Self RAG

> **Tier 4 | Agentic & Self-Improving**

## What is Self RAG?

Standard RAG generates one answer and stops. **Self RAG** generates a draft,
then critically evaluates it on four quality dimensions and iteratively refines
until all dimensions pass a threshold or a max-iteration cap is reached.

### Four quality dimensions

| Dimension | Question asked | Threshold |
|-----------|---------------|-----------|
| **Faithfulness** | Is every claim grounded in the retrieved context? | ≥ 0.7 |
| **Completeness** | Does the answer address all parts of the question? | ≥ 0.7 |
| **Conciseness** | Is the answer free of padding and repetition? | ≥ 0.7 |
| **Groundedness** | Are there no hallucinated facts not in the context? | ≥ 0.8 |

If any dimension fails, the critic generates a targeted improvement instruction
and the answer is regenerated from the same context. Up to `MAX_ITERATIONS`
refinement cycles.

## PDF Framework: pymupdf (fitz)

This notebook uses **pymupdf** — the Python binding for MuPDF, a C library.

| Feature | pymupdf advantage |
|---------|------------------|
| Speed | 5–10× faster than pdfplumber for text extraction |
| Page iterator | `doc[i].get_text("dict")` returns structured block/line/span JSON |
| Image extraction | `page.get_images()` |
| Rotation-aware | Handles rotated pages correctly |
| Font metadata | Span-level font name + size → detect headings |

We use `get_text("dict")` to extract **block-level structure** and detect
headings (large or bold font) — storing `is_heading` and `font_size_avg` per
chunk, which the self-critic can use when assessing completeness.

## Flow Diagram

```mermaid
flowchart TD
    subgraph INDEX ["🔵  INDEXING — pymupdf"]
        PDF(["📄 climate.pdf"])
        PDF --> FZ["fitz.open()\nget_text('dict')"]
        FZ --> BLOCKS["Block-level\nstructure + font metadata"]
        BLOCKS --> SPLIT["Chunk with\nis_heading, font_size_avg"]
        SPLIT --> EMB["Embed — Titan V2"]
        EMB --> QDRANT[("Qdrant\nself_rag_19")]
    end

    subgraph LOOP ["🔄  SELF-CRITIQUE LOOP (max N iterations)"]
        Q(["❓ Query"])
        Q --> RET["Retrieve top-K"]
        QDRANT --> RET
        RET --> GEN["Generate draft answer"]
        GEN --> CRITIC["Self-Critic:\nFaithfulness · Completeness\nConciseness · Groundedness"]
        CRITIC --> CHK{{"All dims\n≥ threshold?"}}
        CHK -->|"No — iteration < N"| INSTRUCT["Targeted improvement\ninstruction"]
        INSTRUCT --> GEN
        CHK -->|"Yes or max reached"| FINAL(["✅ Final answer\n+ quality scores"])
    end

    style INDEX fill:#dbeafe,stroke:#3b82f6,color:#1e3a5f
    style LOOP  fill:#fef9c3,stroke:#ca8a04,color:#713f12
```


In [ ]:
from IPython.display import display as _d, HTML as _H
_d(_H("""
<script src="https://cdn.jsdelivr.net/npm/mermaid@10/dist/mermaid.min.js"></script>
<div class="mermaid" style="background:#fff;padding:16px;border-radius:8px;">
flowchart TD
    subgraph INDEX ["🔵  INDEXING — pymupdf"]
        PDF(["📄 climate.pdf"])
        PDF --> FZ["fitz.open()\nget_text('dict')"]
        FZ --> BLOCKS["Block-level\nstructure + font metadata"]
        BLOCKS --> SPLIT["Chunk with\nis_heading, font_size_avg"]
        SPLIT --> EMB["Embed — Titan V2"]
        EMB --> QDRANT[("Qdrant\nself_rag_19")]
    end

    subgraph LOOP ["🔄  SELF-CRITIQUE LOOP (max N iterations)"]
        Q(["❓ Query"])
        Q --> RET["Retrieve top-K"]
        QDRANT --> RET
        RET --> GEN["Generate draft answer"]
        GEN --> CRITIC["Self-Critic:\nFaithfulness · Completeness\nConciseness · Groundedness"]
        CRITIC --> CHK{{"All dims\n≥ threshold?"}}
        CHK -->|"No — iteration < N"| INSTRUCT["Targeted improvement\ninstruction"]
        INSTRUCT --> GEN
        CHK -->|"Yes or max reached"| FINAL(["✅ Final answer\n+ quality scores"])
    end

    style INDEX fill:#dbeafe,stroke:#3b82f6,color:#1e3a5f
    style LOOP  fill:#fef9c3,stroke:#ca8a04,color:#713f12
</div>
<script>mermaid.initialize({startOnLoad:true,theme:'default',flowchart:{curve:'basis'}});</script>
"""))


## Step 1 — Install & Import Dependencies

In [ ]:
import subprocess, sys
packages = ["boto3", "qdrant-client", "opensearch-py", "requests-aws4auth",
            "strands-agents", "pymupdf", "python-dotenv"]
subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet"] + packages)
print("All packages ready.")


In [ ]:
import os, json, time, uuid, re
from typing import List, Dict, Tuple, Optional
from dotenv import load_dotenv

import boto3, fitz   # fitz is the pymupdf import name
from strands import Agent
from strands.models.bedrock import BedrockModel
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams, PointStruct

load_dotenv(r"C:\Users\Administrator\RAG\.env", override=True)
print(f"pymupdf version : {fitz.__version__}")
print("Imports OK")


## Step 2 — Configuration

In [ ]:
# AWS / Bedrock
AWS_REGION      = os.getenv("AWS_DEFAULT_REGION", "us-east-1")
EMBEDDING_MODEL = "amazon.titan-embed-text-v2:0"
LLM_MODEL       = "us.anthropic.claude-sonnet-4-6"

# Vector DB
QDRANT_URL     = os.getenv("QDRANT_URL", "")
QDRANT_API_KEY = os.getenv("QDRANT_API_KEY", "")
OPENSEARCH_URL = os.getenv("OPENSEARCH_ENDPOINT", "")

# Collection
COLLECTION_NAME = "self_rag_19"
EMBEDDING_DIM   = 1024
TOP_K           = 5

# Chunking
CHUNK_SIZE    = 500
CHUNK_OVERLAP = 50

# Self-RAG quality thresholds (0.0 – 1.0)
THRESHOLDS = {
    'faithfulness' : 0.7,
    'completeness' : 0.7,
    'conciseness'  : 0.7,
    'groundedness' : 0.8,
}
MAX_ITERATIONS = 3   # max refinement cycles before returning best answer

# PDF
PDF_PATH = r"C:\Users\Administrator\RAG\data\climate.pdf"

print(f"Collection  : {COLLECTION_NAME}")
print(f"PDF         : {PDF_PATH}  (exists={os.path.exists(PDF_PATH)})")
print(f"Thresholds  : {THRESHOLDS}")
print(f"Max iters   : {MAX_ITERATIONS}")


## Step 3 — Vector Store

In [ ]:
class VectorStore:
    def __init__(self, collection_name, qdrant_url='', qdrant_api_key='',
                 opensearch_url='', region='us-east-1'):
        self.name     = collection_name
        self._backend = None
        if qdrant_url:
            try:
                kw = {'url': qdrant_url}
                if qdrant_api_key: kw['api_key'] = qdrant_api_key
                self._qdrant = QdrantClient(**kw)
                self._qdrant.get_collections()
                self._backend = 'qdrant_cloud'
                print(f'Qdrant Cloud: {qdrant_url}')
                return
            except Exception as e:
                print(f'Qdrant Cloud unavailable ({e}), trying next...')
        if opensearch_url:
            try:
                from opensearchpy import OpenSearch, RequestsHttpConnection, AWSV4SignerAuth
                creds = boto3.Session().get_credentials()
                auth  = AWSV4SignerAuth(creds, region, 'aoss')
                host  = opensearch_url.replace('https://','').replace('http://','')
                self._os = OpenSearch(
                    hosts=[{'host': host, 'port': 443}],
                    http_auth=auth, use_ssl=True, verify_certs=True,
                    connection_class=RequestsHttpConnection, timeout=30)
                self._os.info()
                self._backend = 'opensearch'
                print(f'OpenSearch: {opensearch_url}')
                return
            except Exception as e:
                print(f'OpenSearch unavailable ({e}), falling back...')
        self._qdrant  = QdrantClient(':memory:')
        self._backend = 'qdrant_memory'
        print('Using Qdrant in-memory')

    def create_collection(self, dim=1024, force_recreate=False):
        if self._backend in ('qdrant_cloud', 'qdrant_memory'):
            exists = self.name in [c.name for c in self._qdrant.get_collections().collections]
            if exists and force_recreate:
                self._qdrant.delete_collection(self.name); exists = False
            if not exists:
                self._qdrant.create_collection(
                    self.name, vectors_config=VectorParams(size=dim, distance=Distance.COSINE))
                print(f'Created "{self.name}" (dim={dim})')
            else:
                print(f'"{self.name}" already exists')

    def upsert(self, docs: List[Dict]) -> int:
        if self._backend in ('qdrant_cloud', 'qdrant_memory'):
            pts = [PointStruct(id=str(uuid.uuid4()), vector=d['embedding'],
                               payload={'text': d['text'], 'metadata': d.get('metadata', {})})
                   for d in docs]
            self._qdrant.upsert(collection_name=self.name, points=pts)
            return len(pts)

    def search(self, qvec: List[float], top_k: int = 5) -> List[Dict]:
        if self._backend in ('qdrant_cloud', 'qdrant_memory'):
            resp = self._qdrant.query_points(
                collection_name=self.name, query=qvec, limit=top_k, with_payload=True)
            return [{'text': p.payload.get('text', ''),
                     'metadata': p.payload.get('metadata', {}),
                     'score': p.score, 'id': str(p.id)}
                    for p in resp.points]

    def count(self) -> int:
        if self._backend in ('qdrant_cloud', 'qdrant_memory'):
            return self._qdrant.get_collection(self.name).points_count or 0

    def delete_collection(self):
        if self._backend in ('qdrant_cloud', 'qdrant_memory'):
            self._qdrant.delete_collection(self.name)
        print(f'Deleted "{self.name}"')

print("VectorStore defined.")


## Step 4 — Bedrock Helpers

In [ ]:
bedrock_rt = boto3.client('bedrock-runtime', region_name=AWS_REGION)

def embed_text(text: str) -> List[float]:
    body = json.dumps({"inputText": text, "dimensions": EMBEDDING_DIM, "normalize": True})
    resp = bedrock_rt.invoke_model(
        modelId=EMBEDDING_MODEL, body=body,
        contentType="application/json", accept="application/json")
    return json.loads(resp["body"].read())["embedding"]

def embed_batch(texts: List[str], label: str = '') -> List[List[float]]:
    out = []
    for i, t in enumerate(texts):
        out.append(embed_text(t))
        if (i + 1) % 20 == 0:
            print(f'  {label} {i+1}/{len(texts)}')
        time.sleep(0.04)
    return out

_model = BedrockModel(model_id=LLM_MODEL, region_name=AWS_REGION)

def llm(prompt: str, system: str = "You are a helpful assistant.") -> str:
    return str(Agent(model=_model, system_prompt=system)(prompt))

test_emb = embed_text("self rag pymupdf test")
print(f"Embedding OK — dim={len(test_emb)}")
print("Strands BedrockModel ready.")


## Step 5 — PDF Loading with pymupdf (fitz)

`fitz.open()` + `get_text("dict")` returns a nested JSON of blocks → lines → spans.
Each span carries `size` (font size) and `flags` (bold = flags & 16).
We aggregate these to compute `font_size_avg` and `is_heading` per chunk —
useful signals for the completeness critic.


In [ ]:
def recursive_split(text: str, chunk_size: int, chunk_overlap: int) -> List[str]:
    separators = ["\n\n", "\n", ". ", " ", ""]
    def _split(text, seps):
        sep, new_seps = '', []
        for i, s in enumerate(seps):
            if s == '' or s in text:
                sep, new_seps = s, seps[i+1:]; break
        parts = text.split(sep) if sep != '' else list(text)
        good = []
        for part in parts:
            if len(part) <= chunk_size: good.append(part)
            elif new_seps: good.extend(_split(part, new_seps))
            else:
                for k in range(0, len(part), chunk_size - chunk_overlap):
                    good.append(part[k:k+chunk_size])
        chunks, cur_pieces, cur_len = [], [], 0
        for piece in good:
            p = piece.strip()
            if not p: continue
            addition = len(sep) + len(p) if cur_pieces else len(p)
            if cur_len + addition <= chunk_size:
                cur_pieces.append(p); cur_len += addition
            else:
                if cur_pieces: chunks.append(sep.join(cur_pieces))
                ov, ovl = [], 0
                for prev in reversed(cur_pieces):
                    if ovl + len(prev) + len(sep) <= chunk_overlap:
                        ov.insert(0, prev); ovl += len(prev) + len(sep)
                    else: break
                cur_pieces = ov + [p]
                cur_len = sum(len(x) + len(sep) for x in cur_pieces)
        if cur_pieces: chunks.append(sep.join(cur_pieces))
        return [c for c in chunks if c.strip()]
    return _split(text, separators)


def load_pdf_pymupdf(path: str) -> Tuple[List[Dict], Dict]:
    """
    Returns:
      chunks     — list of {text, page_num, font_size_avg, is_heading, char_count}
      stats      — extraction summary
    """
    doc    = fitz.open(path)
    chunks = []
    total_chars = 0

    for page_num in range(len(doc)):
        page     = doc[page_num]
        page_dict = page.get_text("dict")

        page_text  = ""
        font_sizes = []
        bold_count = 0
        span_count = 0

        for block in page_dict.get("blocks", []):
            if block.get("type") != 0:   # 0 = text block
                continue
            for line in block.get("lines", []):
                for span in line.get("spans", []):
                    span_text = span.get("text", "").strip()
                    if span_text:
                        page_text += span_text + " "
                        font_sizes.append(span.get("size", 12.0))
                        if span.get("flags", 0) & 16:   # bold flag
                            bold_count += 1
                        span_count += 1
                page_text += "\n"

        page_text = page_text.strip()
        if not page_text:
            continue

        font_size_avg = round(sum(font_sizes) / max(len(font_sizes), 1), 2)
        body_size     = sorted(font_sizes)[len(font_sizes) // 2] if font_sizes else 12.0

        page_chunks = recursive_split(page_text, CHUNK_SIZE, CHUNK_OVERLAP)
        for chunk in page_chunks:
            # A chunk is a heading if its font is noticeably larger than median body
            # or if more than half of spans on the page were bold
            is_heading = (font_size_avg > body_size * 1.15) or (bold_count > span_count * 0.5)
            chunks.append({
                'text'         : chunk,
                'page_num'     : page_num + 1,
                'font_size_avg': font_size_avg,
                'is_heading'   : is_heading,
                'char_count'   : len(chunk),
                'word_count'   : len(chunk.split()),
            })
            total_chars += len(chunk)

    doc.close()

    stats = {
        'n_pages'      : len(fitz.open(path)),
        'n_chunks'     : len(chunks),
        'avg_chunk_len': total_chars // max(len(chunks), 1),
        'heading_chunks': sum(1 for c in chunks if c['is_heading']),
    }
    return chunks, stats


t0 = time.time()
chunks, stats = load_pdf_pymupdf(PDF_PATH)
elapsed = (time.time() - t0) * 1000

print(f"pymupdf extraction : {elapsed:.0f}ms")
print(f"Pages              : {stats['n_pages']}")
print(f"Chunks             : {stats['n_chunks']}")
print(f"Avg chunk length   : {stats['avg_chunk_len']} chars")
print(f"Heading chunks     : {stats['heading_chunks']}")
print()
print("Sample chunk (first):")
c0 = chunks[0]
print(f"  page={c0['page_num']}  font_size_avg={c0['font_size_avg']}  "
      f"is_heading={c0['is_heading']}  chars={c0['char_count']}")
print(f"  text: {c0['text'][:120]}...")


## Step 6 — Connect & Index

In [ ]:
vs = VectorStore(
    collection_name=COLLECTION_NAME,
    qdrant_url=QDRANT_URL, qdrant_api_key=QDRANT_API_KEY,
    opensearch_url=OPENSEARCH_URL, region=AWS_REGION)
print(f"Backend: {vs._backend}")

vs.create_collection(dim=EMBEDDING_DIM, force_recreate=True)

print(f"Embedding {len(chunks)} chunks...")
t0   = time.time()
embs = embed_batch([c['text'] for c in chunks], label='[index]')

docs = [
    {
        'text'     : chunks[i]['text'],
        'embedding': embs[i],
        'metadata' : {
            'chunk_idx'    : i,
            'page_num'     : chunks[i]['page_num'],
            'font_size_avg': chunks[i]['font_size_avg'],
            'is_heading'   : chunks[i]['is_heading'],
            'char_count'   : chunks[i]['char_count'],
            'word_count'   : chunks[i]['word_count'],
            'source'       : 'climate.pdf',
            'pdf_lib'      : 'pymupdf',
        }
    }
    for i in range(len(chunks))
]
vs.upsert(docs)
print(f"Indexed {vs.count()} vectors in {time.time()-t0:.1f}s")


## Step 7 — Self-Critic

The critic evaluates a draft answer on all four dimensions in a single LLM call
(returning JSON), then identifies the weakest dimension and writes a targeted
improvement instruction.


In [ ]:
CRITIC_SYSTEM = """You are a rigorous answer quality evaluator.
Given a question, context passages, and a draft answer, score the answer on four dimensions.
Return ONLY valid JSON with this exact structure:
{
  "faithfulness":  <float 0.0-1.0>,
  "completeness":  <float 0.0-1.0>,
  "conciseness":   <float 0.0-1.0>,
  "groundedness":  <float 0.0-1.0>,
  "weakest_dim":   "<dimension name>",
  "improvement":   "<one sentence: what specifically to fix>"
}

Definitions:
- faithfulness:  every claim in the answer is directly supported by the context (1.0 = fully)
- completeness:  the answer addresses all parts of the question (1.0 = fully)
- conciseness:   the answer is tight with no padding or repetition (1.0 = perfectly concise)
- groundedness:  no facts appear that are not in the context (1.0 = zero hallucination)
"""

def critique(question: str, context_passages: List[str], draft: str) -> Dict:
    context = '\n\n'.join(f'[P{i+1}] {p[:300]}' for i, p in enumerate(context_passages))
    prompt  = (
        f"Question: {question}\n\n"
        f"Context:\n{context}\n\n"
        f"Draft answer:\n{draft}\n\n"
        "Evaluate and return JSON:"
    )
    raw = llm(prompt, system=CRITIC_SYSTEM).strip()
    m   = re.search(r'\{[\s\S]+\}', raw)
    if m:
        try:
            return json.loads(m.group())
        except json.JSONDecodeError:
            pass
    # Fallback if JSON parsing fails — assume all dimensions at 0.5
    return {d: 0.5 for d in THRESHOLDS} | {'weakest_dim': 'faithfulness',
                                             'improvement': 'Be more specific and grounded.'}

def passes_all(scores: Dict) -> bool:
    return all(scores.get(d, 0) >= THRESHOLDS[d] for d in THRESHOLDS)

# Unit test
_ctx  = ["Weather forecasting uses numerical models and satellite data."]
_ans  = "Weather forecasting relies on numerical models and satellite observations."
_crit = critique("How is weather forecasting done?", _ctx, _ans)
print("Critic test scores:")
for dim in THRESHOLDS:
    mark = '✓' if _crit.get(dim, 0) >= THRESHOLDS[dim] else '✗'
    print(f"  {mark} {dim:<14}: {_crit.get(dim, 0):.2f}  (thresh={THRESHOLDS[dim]})")
print(f"  Weakest  : {_crit.get('weakest_dim','?')}")
print(f"  Improve  : {_crit.get('improvement','?')}")
print(f"  Passes   : {passes_all(_crit)}")


## Step 8 — Generator

`generate()` produces a draft answer, optionally incorporating an improvement
instruction from the previous critique cycle.


In [ ]:
GEN_SYSTEM = (
    "You are a precise Q&A assistant. Answer only from the provided context. "
    "Be faithful, complete, concise, and grounded. "
    "If the answer is not in the context say 'Not found in context.'"
)

def generate(question: str, passages: List[str], improvement: Optional[str] = None) -> str:
    context = '\n\n'.join(f'[Passage {i+1}]\n{p}' for i, p in enumerate(passages))
    refine_note = (
        f"\n\nIMPORTANT IMPROVEMENT REQUIRED: {improvement}"
        if improvement else ""
    )
    prompt = (
        f"Context:\n{context}\n\n"
        f"Question: {question}{refine_note}\n\n"
        "Answer:"
    )
    return llm(prompt, system=GEN_SYSTEM).strip()


## Step 9 — Full Self RAG Pipeline

`self_rag()` wires together retrieval, generation, and the self-critique loop:

```
retrieve → generate draft → critique → [refine if needed] × MAX_ITERATIONS → return best
```

The loop tracks scores across iterations and returns the answer with the
highest average score if the threshold is never fully met.


In [ ]:
def self_rag(question: str, top_k: int = TOP_K, verbose: bool = True) -> Dict:
    t0 = time.time()

    # ── Retrieve (once — same context for all iterations) ──
    passages = [h['text'] for h in vs.search(embed_text(question), top_k=top_k)]

    if verbose:
        print(f"\nQ: {question}")
        print(f"Retrieved {len(passages)} passages")

    improvement  = None
    history      = []   # [(answer, scores)]
    best_answer  = None
    best_avg     = -1.0

    for iteration in range(1, MAX_ITERATIONS + 1):
        # Generate
        draft = generate(question, passages, improvement)

        # Critique
        scores = critique(question, passages, draft)
        avg    = sum(scores.get(d, 0) for d in THRESHOLDS) / len(THRESHOLDS)

        if verbose:
            flag = '✓' if passes_all(scores) else '✗'
            print(f"\n  Iter {iteration} {flag}  avg={avg:.2f}  "
                  f"faith={scores.get('faithfulness',0):.2f}  "
                  f"comp={scores.get('completeness',0):.2f}  "
                  f"conc={scores.get('conciseness',0):.2f}  "
                  f"grnd={scores.get('groundedness',0):.2f}")
            print(f"  Draft: {draft[:120]}...")

        history.append({'iteration': iteration, 'answer': draft, 'scores': scores, 'avg': avg})

        if avg > best_avg:
            best_avg    = avg
            best_answer = draft

        if passes_all(scores):
            if verbose:
                print(f"  All dimensions passed — stopping at iteration {iteration}")
            break

        # Prepare improvement for next iteration
        improvement = scores.get('improvement', 'Improve faithfulness and groundedness.')
        if verbose:
            print(f"  Weakest: {scores.get('weakest_dim','?')} → {improvement}")

    latency = (time.time() - t0) * 1000

    final_scores = history[-1]['scores'] if history else {}
    if verbose:
        print(f"\nFinal answer ({latency:.0f}ms, {len(history)} iter):")
        print(best_answer)
        print("-" * 70)

    return {
        'question'   : question,
        'answer'     : best_answer,
        'iterations' : len(history),
        'converged'  : passes_all(final_scores),
        'final_scores': {d: round(final_scores.get(d, 0), 3) for d in THRESHOLDS},
        'best_avg'   : round(best_avg, 3),
        'latency_ms' : latency,
        'history'    : history,
    }


# Demo
result = self_rag("What are the main methods used in weather forecasting?")


## Step 10 — Multi-Question Demo

In [ ]:
test_questions = [
    "How do satellites contribute to weather observation?",
    "What is the difference between weather forecasting and climatology?",
    "What instruments are used to measure atmospheric pressure?",
    "What is nowcasting and how does it differ from regular forecasting?",
]

all_results = []
for q in test_questions:
    r = self_rag(q, verbose=True)
    all_results.append(r)

print("\n" + "=" * 80)
print("SUMMARY")
print("=" * 80)
print(f"{'Question':<50} {'Iters':>5}  {'Converged':>9}  {'Best avg':>8}  {'ms':>6}")
print("-" * 80)
for r in all_results:
    print(f"{r['question'][:49]:<50} {r['iterations']:>5}  "
          f"{'yes' if r['converged'] else 'no':>9}  "
          f"{r['best_avg']:>8.3f}  {r['latency_ms']:>6.0f}")


## Step 11 — Score Progression Plot

Show how each quality dimension evolves across iterations for a representative question.


In [ ]:
# Use the first result with more than 1 iteration for an interesting trace,
# otherwise fall back to the first result
trace_result = next((r for r in all_results if r['iterations'] > 1), all_results[0])

print(f"Score progression for: '{trace_result['question']}'")
print()
dims = list(THRESHOLDS.keys())
header = f"{'Iter':>5}  " + "  ".join(f"{d[:6]:>8}" for d in dims) + f"  {'avg':>7}"
print(header)
print("-" * len(header))

for h in trace_result['history']:
    scores_str = "  ".join(f"{h['scores'].get(d, 0):>8.3f}" for d in dims)
    converged  = '✓' if passes_all(h['scores']) else ' '
    print(f"{h['iteration']:>4}{converged}  {scores_str}  {h['avg']:>7.3f}")

print()
print(f"Threshold: " + "  ".join(f"{THRESHOLDS[d]:>8.1f}" for d in dims))
print(f"Converged : {trace_result['converged']}")
print(f"Best avg  : {trace_result['best_avg']:.3f}")


## Step 12 — Self RAG vs Simple RAG Comparison

In [ ]:
def simple_rag(question: str, top_k: int = TOP_K) -> str:
    passages = [h['text'] for h in vs.search(embed_text(question), top_k=top_k)]
    return generate(question, passages)

eval_qs = [
    {"q": "What are the types of weather forecasts by time range?",
     "kw": ["short", "medium", "long", "range", "nowcast"]},
    {"q": "How does numerical weather prediction work?",
     "kw": ["numerical", "model", "equation", "computer", "atmosphere"]},
    {"q": "What is the role of synoptic charts in meteorology?",
     "kw": ["synoptic", "chart", "pressure", "isobar", "map"]},
]

print(f"{'Question':<50} {'Simple':>8}  {'SelfRAG':>8}  {'Iters':>5}")
print("-" * 76)

for eq in eval_qs:
    q   = eq['q']
    kws = eq['kw']
    n   = len(kws)

    s_ans = simple_rag(q)
    sr    = self_rag(q, verbose=False)

    s_hit = sum(1 for kw in kws if kw in s_ans.lower())
    c_hit = sum(1 for kw in kws if kw in sr['answer'].lower())

    print(f"{q[:49]:<50} {s_hit}/{n} ({s_hit/n*100:.0f}%)  "
          f"{c_hit}/{n} ({c_hit/n*100:.0f}%)  {sr['iterations']:>5}")

print()
print("Self RAG adds critique overhead but refines until quality thresholds are met.")


## Step 13 — Summary

| Component | Implementation |
|-----------|---------------|
| PDF loading | **pymupdf (fitz)** — `get_text("dict")` block/span structure |
| Font metadata | `font_size_avg`, `is_heading` extracted per page, stored in payload |
| Chunking | Native `recursive_split()` — 500 chars, 50 overlap |
| Embeddings | Amazon Bedrock Titan V2 (1024-dim) |
| Generator | Strands Agent + Claude Sonnet 4.6 (with optional improvement instruction) |
| Self-Critic | Single LLM call → JSON scores on 4 dimensions + improvement hint |
| Loop | Up to `MAX_ITERATIONS=3` refinement cycles; returns best-avg answer |
| Convergence | All 4 dimensions ≥ threshold → early stop |
| Vector DB | Qdrant Cloud → OpenSearch → in-memory |

## pymupdf vs pdfplumber — key difference

| | pdfplumber (nb 18) | pymupdf (nb 19) |
|--|-------------------|-----------------|
| Speed | ~2–3× slower | **~5–10× faster** (C/MuPDF core) |
| Per-word bbox | ✓ | ✓ via `get_text("dict")` |
| Font metadata | ✗ | ✓ **size + bold flags per span** |
| Table extraction | ✓ | Needs extra call |
| Best for | Metadata-rich indexing | **Iterative loops needing speed** |

### PDF Framework progression

| NB | Framework | Key advantage |
|----|-----------|--------------|
| 01–17 | pypdf | Simple text |
| 18 | pdfplumber | bbox density + tables |
| **19** | **pymupdf** | **Speed + font/heading metadata** |
| 20 | pdfminer.six | Fine-grained layout tree |
| 21 | unstructured | Typed elements |
| 22 | docling | Structured JSON |
| 29 | All 6 | Full benchmark |

### Next: **20 — Iterative RAG** (pdfminer.six)


In [ ]:
# vs.delete_collection()  # uncomment to clean up
print(f"Collection '{COLLECTION_NAME}' in {vs._backend} — {vs.count()} vectors")
print(f"PDF framework  : pymupdf (fitz {fitz.__version__})")
print(f"Chunks indexed : {stats['n_chunks']}  (heading chunks: {stats['heading_chunks']})")
print(f"Thresholds     : {THRESHOLDS}")
print(f"Max iterations : {MAX_ITERATIONS}")
print("\nNotebook 19 complete. Give the go-ahead for notebook 20 (Iterative RAG / pdfminer.six).")
